In [ ]:
#@title Setup
# DATA_DIR: folder with the input tables from Dryad (https://doi.org/10.5061/dryad.1vhhmgr88). The fine-tuning panels
# also need the ProteinGym substitution benchmark there (DMS_substitutions.csv and the DMS/ folder of assays).
# Figures and intermediate tables are written to the same folder. On Colab, mount Google Drive first if the data are there.
# Figures use Arial: put arial.ttf in DATA_DIR (Arial cannot be redistributed), otherwise the default sans-serif font is used.
!pip install -q pypalettes

import os
import matplotlib.font_manager as fm

DATA_DIR = "HuSC_data"
os.chdir(DATA_DIR)
ARIAL = "arial.ttf" if os.path.exists("arial.ttf") else fm.findfont("DejaVu Sans")

# Plot settings

In [ ]:
import warnings
warnings.filterwarnings("ignore", category=DeprecationWarning)

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import pypalettes as pp

In [ ]:
import warnings
warnings.filterwarnings("ignore", category=DeprecationWarning)

def apply_global_font(ax, size=26):
    """Apply Arial font and consistent size to axes labels and ticks."""
    for label in ax.get_xticklabels() + ax.get_yticklabels():
        label.set_fontproperties(arial_font)
        label.set_fontsize(size)
    for axis_label in [ax.xaxis.label, ax.yaxis.label]:
        axis_label.set_fontproperties(arial_font)
        axis_label.set_fontsize(size)

In [ ]:
# --- Global Plot Settings ---
import warnings
warnings.filterwarnings("ignore", category=DeprecationWarning)

import matplotlib.pyplot as plt
import seaborn as sns
import matplotlib.font_manager as fm

# Load font
arial_font = fm.FontProperties(fname=ARIAL)

# --- Global rcParams ---
plt.rcParams.update({
    # Font and text
    'font.family': 'sans-serif',
    'font.sans-serif': ['Arial'],
    'font.size': 20,
    'axes.titlesize': 24,
    'axes.labelsize': 26,
    'axes.labelweight': 'bold',
    'xtick.labelsize': 22,
    'ytick.labelsize': 22,
    'axes.linewidth': 2.5,

    # Remove gridlines globally
    'axes.grid': False,
    'grid.linestyle': '',

    # Axes and ticks (ensures they are visible)
    'axes.edgecolor': 'black',
    'xtick.bottom': True,
    'ytick.left': True,
    'xtick.top': False,
    'ytick.right': False,
    'xtick.color': 'black',
    'ytick.color': 'black',
    'xtick.direction': 'out',
    'ytick.direction': 'out',
    'xtick.major.width': 2.5,
    'ytick.major.width': 2.5,
    'xtick.major.size': 20,
    'ytick.major.size': 20,

    # Figure defaults
    'figure.figsize': (8, 8),
    'savefig.format': 'svg',
    'savefig.bbox': 'tight',
    'savefig.dpi': 300,

    # Lines
    'lines.linewidth': 2,

    # Legend
    'legend.fontsize': 20,
    'legend.frameon': False
})

# Clean Seaborn base theme (no gridlines)
sns.set_theme(style="white", rc=plt.rcParams)

# Figures

In [ ]:
#@title Fig 2A

import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
from pypalettes import load_cmap

# --- Configuration  ---
FIGURE_SIZE = (6, 6)
DPI = 300
FONT_SIZE_LABELS = 18
FONT_SIZE_TICKS = 18
FONT_SIZE_LEGEND = 18
FONT_SIZE_TEXT = 15
SPINE_WIDTH = 1.2

# Load Arial font
arial_font = fm.FontProperties(fname=ARIAL)

# Load color palette
colors = load_cmap('Royal1').colors

# --- Load data ---
df = pd.read_csv('all_husc_values_2026-10.txt', header=None, names=['HuSC'])
print(f"Total number of residues:{len(df)}")

df['HuSC'] = pd.to_numeric(df['HuSC'], errors='coerce')

# HuSC is bounded by the permutation count (10,000 permutations, mid-p: +-3.891);
# clip the KDE to the observed range so no density is drawn past the bounds
husc_bounds = (df['HuSC'].min(), df['HuSC'].max())
print(f"HuSC range: {husc_bounds}; at lower bound: {(df['HuSC'] == husc_bounds[0]).mean():.2%}, "
      f"at upper bound: {(df['HuSC'] == husc_bounds[1]).mean():.2%}")

# --- Create figure  ---
fig, ax = plt.subplots(figsize=FIGURE_SIZE, dpi=DPI)
ax.set_box_aspect(1)  # square plotting area, independent of tick-label widths

# KDE plot
sns.kdeplot(df['HuSC'], color=colors[1], fill=True, alpha=0.3,
            linewidth=2.5, clip=husc_bounds, ax=ax)

# Leave headroom above the density peak for the annotation
y_top = ax.get_ylim()[1] * 1.12
ax.set_ylim(0, y_top)

# Mean, annotated next to the line
mean_val = df['HuSC'].mean()
print(f"Mean HuSC: {mean_val:.3f}; median: {df['HuSC'].median():.3f}; SD: {df['HuSC'].std():.3f}")
ax.axvline(mean_val, color=colors[1], linestyle='--', linewidth=2, alpha=0.8)
ax.text(mean_val + 0.12, y_top * 0.96, f"Mean = {mean_val:.2f}".replace('-', '−'),
        ha='left', va='top', color='black', fontproperties=arial_font, fontsize=FONT_SIZE_TEXT)

# Labels and formatting (matching Fig 6B)
ax.set_xlabel("HuSC", fontsize=FONT_SIZE_LABELS,
              fontweight='bold', fontproperties=arial_font)
ax.set_ylabel("Density", fontsize=FONT_SIZE_LABELS,
              fontweight='bold', fontproperties=arial_font)

# Tick formatting
ax.tick_params(axis='both', which='major', labelsize=FONT_SIZE_TICKS, width=SPINE_WIDTH, bottom=True, left=True)

for label in ax.get_xticklabels() + ax.get_yticklabels():
    label.set_fontproperties(arial_font)
    label.set_fontsize(FONT_SIZE_TICKS)

# Spines
for spine in ['top', 'right']:
    ax.spines[spine].set_visible(False)
for spine in ['left', 'bottom']:
    ax.spines[spine].set_linewidth(SPINE_WIDTH)
    ax.spines[spine].set_color('black')

plt.tight_layout()

# Save the figure
plt.savefig('HuSC_density_2026-10.svg', dpi=DPI, bbox_inches='tight')

plt.show()

In [ ]:
#@title Fig 2C
# gene_set_scores_HuSC_8_2026-10.tsv: HuSC of every residue of the proteins in each gene class
# (gene class lists from COSMIS, Li et al. 2022).

import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from pypalettes import load_cmap
import matplotlib.font_manager as fm

# ===== Configuration =====
FIGURE_SIZE = (12, 6)
DPI = 300
FONT_SIZE_LABELS = 18
FONT_SIZE_TICKS = 18
FONT_SIZE_TEXT = 18
SPINE_WIDTH = 1.2

# ===== Load Arial font =====
arial_font = fm.FontProperties(fname=ARIAL)

# ===== Load Data =====
df = pd.read_csv('gene_set_scores_HuSC_8_2026-10.tsv', sep='\t', usecols=['class', 'HuSC', 'uniprot_id'])

# ===== Print Summary Statistics =====
print("=" * 80)
print("SUMMARY STATISTICS BY CLASS")
print("=" * 80)

classes = ['Olfactory', 'Non essential', 'Recessive', 'Dominant', 'Essential', 'Haploinsufficient']
# Violins ordered from highest to lowest median HuSC
class_order = df.groupby('class')['HuSC'].median().reindex(classes).sort_values(ascending=False).index.tolist()

for cls in class_order:
    class_data = df[df['class'] == cls]['HuSC']
    if len(class_data) > 0:
        print(f"\n{cls}:")
        print(f"  Count:   {len(class_data)}")
        print(f"  Genes:   {df.loc[df['class'] == cls, 'uniprot_id'].nunique()}")
        print(f"  Mean:    {class_data.mean():.4f}")
        print(f"  Median:  {class_data.median():.4f}")
        print(f"  Std:     {class_data.std():.4f}")
        print(f"  Min:     {class_data.min():.4f}")
        print(f"  Max:     {class_data.max():.4f}")
        print(f"  Q1:      {class_data.quantile(0.25):.4f}")
        print(f"  Q3:      {class_data.quantile(0.75):.4f}")

print("\n" + "=" * 80)
print("OVERALL STATISTICS")
print("=" * 80)
print(df.groupby('class')['HuSC'].describe().reindex(class_order))
print("=" * 80 + "\n")

# ===== Define color palette (assigned by plotting position) =====
palette = load_cmap("AsteroidCity2").colors[:len(class_order)]

# ===== Create Figure =====
fig, ax = plt.subplots(figsize=FIGURE_SIZE, dpi=DPI)

# ===== Plot =====
sns.violinplot(
    x='class', y='HuSC', data=df,
    order=class_order, hue='class', hue_order=class_order, palette=palette, legend=False,
    cut=0, inner="box", linewidth=SPINE_WIDTH+0.5, ax=ax
)

# ===== Axis Labels =====
ax.set_xlabel('Class', fontsize=FONT_SIZE_LABELS,
              fontweight='bold', fontproperties=arial_font, labelpad=10)
ax.set_ylabel('HuSC', fontsize=FONT_SIZE_LABELS,
              fontweight='bold', fontproperties=arial_font, labelpad=10)

# ===== Annotation =====
# Count number of rows (sites) per class
site_counts = df['class'].value_counts().reindex(class_order)

ymax = df['HuSC'].max()
yoffset = 0.05 * (df['HuSC'].max() - df['HuSC'].min())

ax.text(-0.6, ymax + yoffset, "# Sites:",
        ha='right', va='bottom', fontsize=FONT_SIZE_TEXT,
        fontweight='bold', fontproperties=arial_font)

for i, cls in enumerate(class_order):
    if not pd.isna(site_counts[cls]):
        ax.text(i, ymax + yoffset, f"{int(site_counts[cls])}",
                ha='center', va='bottom', fontsize=FONT_SIZE_TEXT,
                fontweight='bold', fontproperties=arial_font)

# ===== Aesthetics =====
# Horizontal line at y=0
ax.axhline(0, color='black', linestyle='--', linewidth=SPINE_WIDTH)

# Tick parameters
ax.tick_params(axis='both', which='major', width=SPINE_WIDTH, color='black',
               bottom=True, left=True)

# ===== Tick label formatting =====
for label in ax.get_xticklabels() + ax.get_yticklabels():
    label.set_fontproperties(arial_font)
    label.set_fontsize(FONT_SIZE_TICKS)

# ===== Spines =====
for spine in ['top', 'right']:
    ax.spines[spine].set_visible(False)
for spine in ['left', 'bottom']:
    ax.spines[spine].set_linewidth(SPINE_WIDTH)
    ax.spines[spine].set_color('black')

# ===== Save and Show =====
plt.tight_layout()
plt.savefig('violin_plot_HuSC_sites_2026-10.svg', dpi=DPI, bbox_inches='tight')
plt.show()

In [ ]:
#@title Fig 3A
# all_conservation_scores_with_HuSC_2026-10.csv: per-residue HuSC and the other conservation and constraint scores.

import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np
from pypalettes import load_cmap
import matplotlib.font_manager as fm

# --- Configuration (matching publication style) ---
FIGURE_SIZE = (10, 8)
DPI = 300
FONT_SIZE_LABELS = 18
FONT_SIZE_TICKS = 18
FONT_SIZE_ANNOT = 15
FONT_SIZE_CBAR_LABEL = 18
FONT_SIZE_CBAR_TICKS = 18
SPINE_WIDTH = 1.2

# Load Arial font
arial_font = fm.FontProperties(fname=ARIAL)

# Load palette
colors = load_cmap('heat').colors

# Load data
all_scores = pd.read_csv('all_conservation_scores_with_HuSC_2026-10.csv.gz', header=0)

# Scores to include
scores = ['HuSC', 'cosmis', 'mtr3d', 'mtr', 'gerp', 'phylop', 'phastcons', 'consurf', 'rvis', 'pli', 'mis_z']

# Spearman correlation (absolute values)
corr = all_scores[scores].corr(method='spearman').abs()
print("HuSC |Spearman| with each score:")
print(corr['HuSC'].drop('HuSC').round(3).to_string())

# Custom reorder
order = ['HuSC', 'cosmis', 'mtr', 'mtr3d', 'rvis', 'pli', 'mis_z',
         'phylop', 'consurf', 'gerp', 'phastcons']
corr_reordered = corr.reindex(index=order, columns=order)

# Pretty labels
labels_ordered = ['HuSC', 'COSMIS', 'MTR', 'MTR3D', 'RVIS', 'pLI', 'Missense Z',
                  'phyloP', 'ConSurf', 'GERP++', 'phastCons']

# Define label colors
label_colors = []
for i, label in enumerate(labels_ordered):
    if label in ['HuSC', 'COSMIS', 'MTR', 'MTR3D']:
        label_colors.append(colors[0])
    elif label in ['RVIS', 'pLI', 'Missense Z']:
        label_colors.append(colors[1])
    else:
        label_colors.append(colors[2])

# Create figure
fig, ax = plt.subplots(figsize=FIGURE_SIZE, dpi=DPI)

# Plot heatmap
sns.heatmap(
    corr_reordered,
    annot=True,
    fmt=".2f",
    cmap='Reds',
    cbar_kws={'label': 'Spearman Correlation'},
    square=True,
    xticklabels=labels_ordered,
    yticklabels=labels_ordered,
    annot_kws={"fontsize": FONT_SIZE_ANNOT, "fontproperties": arial_font},
    linewidths=0,
    linecolor='white',
    ax=ax
)

# Format colorbar
cbar = ax.collections[0].colorbar
cbar.ax.yaxis.label.set_fontproperties(arial_font)
cbar.ax.yaxis.label.set_size(FONT_SIZE_CBAR_LABEL)
cbar.ax.yaxis.label.set_weight('bold')
cbar.outline.set_linewidth(SPINE_WIDTH)
cbar.outline.set_edgecolor('black')

for tick in cbar.ax.get_yticklabels():
    tick.set_fontproperties(arial_font)
    tick.set_fontsize(FONT_SIZE_CBAR_TICKS)

# Rotate and format tick labels
ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha='right')
ax.set_yticklabels(ax.get_yticklabels(), rotation=0)

# Apply font and colors to tick labels
for xtick, color in zip(ax.get_xticklabels(), label_colors):
    xtick.set_fontproperties(arial_font)
    xtick.set_fontsize(FONT_SIZE_TICKS)
    xtick.set_color(color)
    xtick.set_weight('bold')

for ytick, color in zip(ax.get_yticklabels(), label_colors):
    ytick.set_fontproperties(arial_font)
    ytick.set_fontsize(FONT_SIZE_TICKS)
    ytick.set_color(color)
    ytick.set_weight('bold')

# Format heatmap spines
for spine in ax.spines.values():
    spine.set_linewidth(SPINE_WIDTH)
    spine.set_edgecolor('black')

plt.tight_layout()

# Save the figure
plt.savefig('heatmap_2026-10.svg', dpi=DPI, bbox_inches='tight', format='svg')

plt.show()

In [ ]:
#@title Fig 3B
# context_clinvar_with_HuSC_2026-10.csv: ClinVar pathogenic and benign missense variants with HuSC and the other scores.
# Bars: AUC on all variants; error bars: 95% protein bootstrap CI (proteins resampled with replacement, 1,000 draws,
# shared across methods). Paired significance tests: supplementary notebook, "S3 stats".

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import roc_auc_score
from pypalettes import load_cmap
from matplotlib import font_manager as fm

# --- Configuration ---
FIGURE_SIZE = (6, 6)
DPI = 300
FONT_SIZE_LABELS = 18
FONT_SIZE_TICKS = 18
SPINE_WIDTH = 1.2
N_BOOT = 1000

# Load Arial font
arial_font = fm.FontProperties(fname=ARIAL)

# Load color palette
colors = load_cmap('Royal1').colors

# --- Load data ---
df = pd.read_csv('context_clinvar_with_HuSC_2026-10.csv', header=0)

# Column mapping for plotting
column_labels = {
    'HuSC': 'HuSC',
    'consurf': 'ConSurf',
    'phylop': 'PhyloP',
    'cosmis': 'COSMIS',
    'gerp': 'GERP',
    'phastcons': 'PhastCons',
    'mtr3d': 'MTR3D'
}

columns_to_plot = list(column_labels.keys())
label_column = 'label'

# Drop missing data
df = df.dropna(subset=columns_to_plot + [label_column]).reset_index(drop=True)

print("Pathogenic variants:", len(df[df['label'] == 1]))
print("Benign variants:", len(df[df['label'] == 0]))
print("Proteins:", df['uniprot_id'].nunique())

# --- Invert scores as needed ---
for col in ['HuSC', 'consurf', 'cosmis', 'mtr3d']:
    df[col] = -df[col]

# --- Protein bootstrap: resample proteins with replacement, same draws for every method ---
y = df[label_column].values
groups = [g.index.values for _, g in df.groupby('uniprot_id')]
rng = np.random.default_rng(42)
draws = []
for _ in range(N_BOOT):
    idx = np.concatenate([groups[k] for k in rng.integers(0, len(groups), len(groups))])
    if len(np.unique(y[idx])) == 2:
        draws.append(idx)

roc_results = {}
for col in columns_to_plot:
    s = df[col].values
    boots = [roc_auc_score(y[i], s[i]) for i in draws]
    roc_results[col] = (roc_auc_score(y, s), *np.percentile(boots, [2.5, 97.5]))

# --- Print AUC values ---
print(f"\n=== ROC AUC (95% protein bootstrap CI, {len(draws)} resamples) ===")
for col in columns_to_plot:
    auc, lo, hi = roc_results[col]
    print(f"{column_labels[col]:12s}: {auc:.4f} [{lo:.4f}, {hi:.4f}]")

# --- Prepare results ---
roc_means = np.array([roc_results[col][0] for col in columns_to_plot])
roc_lo = np.array([roc_results[col][1] for col in columns_to_plot])
roc_hi = np.array([roc_results[col][2] for col in columns_to_plot])

# Sort by AUC (descending)
roc_order = np.argsort(-roc_means)

# --- Create figure ---
fig, ax = plt.subplots(figsize=FIGURE_SIZE, dpi=DPI)

x = np.arange(len(columns_to_plot))

ax.bar(x, roc_means[roc_order],
       yerr=[roc_means[roc_order] - roc_lo[roc_order], roc_hi[roc_order] - roc_means[roc_order]],
       capsize=4, width=0.6, color=colors[0])

# Labels and formatting
ax.set_xticks(x)
ax.set_xticklabels([column_labels[columns_to_plot[i]] for i in roc_order],
                   ha='center', rotation=45)
ax.set_ylabel("ROC AUC", fontsize=FONT_SIZE_LABELS,
              fontweight='bold', fontproperties=arial_font)
ax.set_ylim(0.5, 1)

# Tick formatting
ax.tick_params(axis='both', which='major', labelsize=FONT_SIZE_TICKS,
               width=SPINE_WIDTH, bottom=True, left=True)

for label in ax.get_xticklabels() + ax.get_yticklabels():
    label.set_fontproperties(arial_font)
    label.set_fontsize(FONT_SIZE_TICKS)

# Spines
for spine in ['top', 'right']:
    ax.spines[spine].set_visible(False)
for spine in ['left', 'bottom']:
    ax.spines[spine].set_linewidth(SPINE_WIDTH)
    ax.spines[spine].set_color('black')

plt.tight_layout()

# Save the figure
plt.savefig("roc_auc_barplot_2026-10.svg", dpi=DPI, bbox_inches='tight')

plt.show()

In [ ]:
#@title Fig 4A
# all_conservation_scores_with_HuSC_2026-10.csv: per-residue HuSC and ConSurf. Missing gene names are filled from
# UniProt (uniprot_gene_names_2026-10.tsv), so no protein is dropped.
# Human-specific constrained (HSC): HuSC < -2.326 (one-sided p < 0.01; estimated FDR ~16% among ConSurf > 3 residues,
# using synonymous scores of the same residues as null; Fig. S4A) & ConSurf > 3.
# Coverage filter: HSC residues whose local gnomAD coverage (median AN within +-15 codons) is < 0.5 of the transcript median
# are excluded (hsc_low_coverage_residues_2026-10.tsv): the null is calibrated per
# transcript, so poorly covered exons look constrained. These residues stay in the scatter, fit and correlations.
# Scatter shows a 1,000,000-residue random subsample (fit and correlations use all residues).

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
from matplotlib.colors import TwoSlopeNorm
from scipy.stats import pearsonr, spearmanr
import matplotlib.font_manager as fm

# Configuration
FIGURE_SIZE = (8, 6)
DPI = 300
FONT_SIZE_LABELS = 18
FONT_SIZE_TICKS = 18
FONT_SIZE_LEGEND = 15
FONT_SIZE_TEXT = 15
SCATTER_SIZE = 25
SCATTER_ALPHA = 0.7
SPINE_WIDTH = 1.2

# Load Arial font
arial_font = fm.FontProperties(fname=ARIAL)

# Load and clean data
print("Loading data...")
df = pd.read_csv("all_conservation_scores_with_HuSC_2026-10.csv.gz", usecols=["uniprot_id", "uniprot_pos", "HuSC", "consurf", "gene_name"])
gene_names = pd.read_csv("uniprot_gene_names_2026-10.tsv", sep="\t").set_index("uniprot_id")["gene_name"]
df["gene_name"] = df["gene_name"].fillna(df["uniprot_id"].map(gene_names))

df = df[["uniprot_id", "uniprot_pos", "HuSC", "consurf", "gene_name"]].replace([np.inf, -np.inf], np.nan).dropna()
print(f"Loaded {len(df):,} residues from {df['gene_name'].nunique():,} genes")

# Define thresholds
HUSC_LOW_MAX = -2.326
CONS_HIGH_MIN = 3.0

# Identify human-specific constrained category (full dataset)
hsc = df[(df["HuSC"] < HUSC_LOW_MAX) & (df["consurf"] > CONS_HIGH_MIN)].copy()
print(f"\nHuman-specific constrained before coverage filter: {len(hsc):,} residues, {hsc['gene_name'].nunique():,} genes")
low_cov = pd.read_csv("hsc_low_coverage_residues_2026-10.tsv", sep="\t")
low_key = set(zip(low_cov["uniprot_id"], low_cov["uniprot_pos"]))
is_low = np.array([k in low_key for k in zip(hsc["uniprot_id"], hsc["uniprot_pos"])])
print(f"Excluded for low gnomAD coverage: {is_low.sum()} residues in {hsc.loc[is_low, 'uniprot_id'].nunique()} proteins")
hsc = hsc[~is_low]
print(f"Human-specific constrained: {len(hsc):,} residues, {hsc['gene_name'].nunique():,} genes")

# Save gene lists
gene_sizes = df.groupby("gene_name").size().reset_index(name="n_residues_total")

hsc_genes = hsc.groupby("gene_name").size().reset_index(name="n_residues_in_category")
hsc_genes = hsc_genes.merge(gene_sizes, on="gene_name")
hsc_genes["proportion_in_category"] = hsc_genes["n_residues_in_category"] / hsc_genes["n_residues_total"]
hsc_genes.to_csv("human_specific_constrained_genes_all_2026-10.csv", index=False)

pd.DataFrame({"gene_name": df["gene_name"].drop_duplicates().sort_values()}).to_csv("background_genes_all_2026-10.csv", index=False)
# GO background for Fig 4B: genes that have at least one residue eligible for the HSC category (ConSurf > 3)
pd.DataFrame({"gene_name": df.loc[df["consurf"] > CONS_HIGH_MIN, "gene_name"].drop_duplicates().sort_values()}).to_csv(
    "background_genes_consurf3_2026-10.csv", index=False)

# Calculate linear fit on full dataset
coeffs = np.polyfit(df["HuSC"], df["consurf"], 1)
m, b = coeffs
df["fitted"] = m * df["HuSC"] + b
df["residuals"] = df["consurf"] - df["fitted"]

# Calculate Pearson correlation on full dataset
r_pearson, pval_pearson = pearsonr(df["HuSC"], df["consurf"])
print(f"\nPearson r = {r_pearson:.4f}")
print(f"Pearson p-value = {pval_pearson}")

# Calculate Spearman correlation on full dataset
r_spearman, pval_spearman = spearmanr(df["HuSC"], df["consurf"])
print(f"\nSpearman rho = {r_spearman:.4f}")
print(f"Spearman p-value = {pval_spearman}")
print(f"Linear fit: y = {m:.4f}x + {b:.4f}")

# Format p-values for display
if pval_pearson < 1e-300:
    pval_pearson_str = "p < 1e-300"
elif pval_pearson < 0.001:
    pval_pearson_str = f"p < {pval_pearson:.2e}"
else:
    pval_pearson_str = f"p = {pval_pearson:.3f}"

if pval_spearman < 1e-300:
    pval_spearman_str = "p < 1e-300"
elif pval_spearman < 0.001:
    pval_spearman_str = f"p < {pval_spearman:.2e}"
else:
    pval_spearman_str = f"p = {pval_spearman:.3f}"

# Downsample for visualization
DOWNSAMPLE_SIZE = 1000000
if len(df) > DOWNSAMPLE_SIZE:
    df_plot = df.sample(n=DOWNSAMPLE_SIZE, random_state=42)
    print(f"\nDownsampled to {len(df_plot):,} residues for visualization")
else:
    df_plot = df
    print(f"\nUsing all {len(df_plot):,} residues for visualization")
in_box = (df_plot["HuSC"] < HUSC_LOW_MAX) & (df_plot["consurf"] > CONS_HIGH_MIN)
print(f"Plotted residues inside the HSC box: {in_box.sum():,}")

# Create plot
fig, ax = plt.subplots(figsize=FIGURE_SIZE, dpi=DPI)

# Create normalization with 0 centered at white
vmin = df["residuals"].quantile(0.01)
vmax = df["residuals"].quantile(0.99)
norm = TwoSlopeNorm(vmin=vmin, vcenter=0, vmax=vmax)

# Plot downsampled residues colored by residuals (coolwarm gradient)
sc = ax.scatter(
    df_plot["HuSC"],
    df_plot["consurf"],
    c=df_plot["residuals"],
    cmap='coolwarm',
    s=SCATTER_SIZE,
    alpha=SCATTER_ALPHA,
    edgecolors="none",
    norm=norm,
    rasterized=True,
    zorder=1
)

# Add colorbar
cbar = plt.colorbar(sc, ax=ax)
cbar.set_label('Residuals from linear fit', fontsize=FONT_SIZE_LABELS,
               fontweight='bold', fontproperties=arial_font)
cbar.ax.tick_params(labelsize=FONT_SIZE_TICKS)
for label in cbar.ax.get_yticklabels():
    label.set_fontproperties(arial_font)

# Plot regression line with both correlations in legend
x_trend = np.array([df["HuSC"].min(), df["HuSC"].max()])
y_trend = m * x_trend + b
ax.plot(
    x_trend, y_trend,
    color='black',
    linewidth=2,
    label=f'Linear fit (y = {m:.2f}x + {b:.2f})\nSpearman ρ = {r_spearman:.3f}, {pval_spearman_str}',
    zorder=2
)

# Draw dashed box for human-specific constrained region
x_min, y_max = df["HuSC"].min(), df["consurf"].max()
rect_hsc = Rectangle(
    (x_min, CONS_HIGH_MIN),
    HUSC_LOW_MAX - x_min,
    y_max - CONS_HIGH_MIN,
    linewidth=2.5,
    edgecolor='black',
    facecolor="none",
    linestyle='--',
    label=f'Human-specific constrained (n={len(hsc):,})',
    zorder=3
)
ax.add_patch(rect_hsc)

# Styling
ax.set_xlabel("HuSC", fontsize=FONT_SIZE_LABELS,
              fontweight='bold', fontproperties=arial_font)
ax.set_ylabel("ConSurf", fontsize=FONT_SIZE_LABELS,
              fontweight='bold', fontproperties=arial_font)
ax.tick_params(axis='both', which='major', labelsize=FONT_SIZE_TICKS, width=SPINE_WIDTH, bottom=True, left=True)


for label in ax.get_xticklabels() + ax.get_yticklabels():
    label.set_fontproperties(arial_font)
    label.set_fontsize(FONT_SIZE_TICKS)

for spine in ['top', 'right']:
    ax.spines[spine].set_visible(False)
for spine in ['left', 'bottom']:
    ax.spines[spine].set_linewidth(SPINE_WIDTH)
    ax.spines[spine].set_color('black')

arial_font_legend = fm.FontProperties(fname=ARIAL, size=FONT_SIZE_LEGEND)

ax.legend(
    fontsize=FONT_SIZE_LEGEND,
    prop=arial_font_legend,
    frameon=True,
    fancybox=False,
    loc='upper right',
    edgecolor='black',
    framealpha=1
)


plt.tight_layout()
plt.savefig("residue_conservation_scatter_publication_2026-10.png", dpi=DPI, bbox_inches='tight')
plt.show()

print("\n✓ Complete!")

In [ ]:
#@title Fig 4B
# Gene list: all genes with >= 1 human-specific constrained residue (HuSC < -2.326 & ConSurf > 3), written by the Fig 4A cell
# (human_specific_constrained_genes_all_2026-10.csv). Background: genes with >= 1 residue with ConSurf > 3
# (background_genes_consurf3_2026-10.csv), i.e. genes that could contain such residues, so the test is not driven by which genes
# have fast-evolving residues at all. Enrichr GO_Biological_Process_2025 via gseapy, library passed BY NAME (Enrichr's server
# library; gseapy.get_library's copy omits the two largest terms, GO:0006355 and GO:0006357). BH FDR.
# x = odds ratio from Enrichr (term sizes within the background are not returned for the server library).
# Shown: terms with FDR < 0.1. Color = -log10(adjusted P), capped at 5 (the transcription terms are ~11-12).

!pip install -q gseapy

import warnings
warnings.filterwarnings("ignore", category=DeprecationWarning)

import json
import requests
import pandas as pd
import numpy as np
import gseapy as gp
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm

# Enrichr's background-enrichment reply contains Infinity odds ratios (terms whose background genes are all hits);
# Colab's simplejson rejects them, so let requests parse with the standard json module, which accepts them.
requests.models.complexjson = json

# =========================
# Configuration
# =========================
FIGURE_SIZE = (9, 6)
DPI = 300
FONT_SIZE_LABELS = 18
FONT_SIZE_TICKS = 12
FONT_SIZE_TERMS = 15
FONT_SIZE_LEGEND = 18
FONT_SIZE_TITLE = 15
SPINE_WIDTH = 1.2
SCATTER_ALPHA = 0.8
EDGE_WIDTH = 0.6
FDR_MAX = 0.1
COLOR_MIN, COLOR_MAX = 1, 5   # -log10(q) color range; 1 = FDR 0.1 (display cutoff), values above 5 shown at the top color

# Load Arial font
arial_font = fm.FontProperties(fname=ARIAL)

def two_lines(term):
    """Split a term name into two lines at the space that best balances the line lengths."""
    words = term.split()
    if len(words) < 2:
        return term
    best = min(range(1, len(words)),
               key=lambda i: max(len(" ".join(words[:i])), len(" ".join(words[i:]))))
    return " ".join(words[:best]) + "\n" + " ".join(words[best:])

# =========================
# Enrichment
# =========================
GENE_SET_LIB = "GO_Biological_Process_2025"

hsc_genes = pd.read_csv("human_specific_constrained_genes_all_2026-10.csv")
background = pd.read_csv("background_genes_consurf3_2026-10.csv")["gene_name"].tolist()
gene_list = hsc_genes["gene_name"].tolist()
print(f"Running enrichment for {len(gene_list)} human-specific constrained genes against {len(background)} background genes...")

enr = gp.enrichr(gene_list=gene_list, gene_sets=[GENE_SET_LIB], background=background, outdir=None)
df = enr.results.copy()
df.to_csv("GO_Biological_Process_2025_table_constrained_2026-10.txt", sep="\t", index=False)
print(f"{len(df)} terms tested; {sum(df['Adjusted P-value'] < 0.05)} at FDR < 0.05, {sum(df['Adjusted P-value'] < FDR_MAX)} at "
      f"FDR < {FDR_MAX}; saved GO_Biological_Process_2025_table_constrained_2026-10.txt")

# =========================
# Preprocessing
# =========================
df["Overlap_Count"] = df["Genes"].str.count(";") + 1
df["Adjusted P-value"] = pd.to_numeric(df["Adjusted P-value"], errors="coerce")
df = df.dropna(subset=["Adjusted P-value"])
df = df[df["Adjusted P-value"] < FDR_MAX].copy()
df["neglog10_q"] = -np.log10(df["Adjusted P-value"])
df["Term"] = df["Term"].str.replace(r"\s*\(GO:\d+\)", "", regex=True)
print(df[["Term", "Overlap_Count", "Odds Ratio", "Adjusted P-value"]].to_string(index=False))
n_inf = np.isinf(df["Odds Ratio"]).sum()
if n_inf:
    print(f"{n_inf} term(s) with infinite odds ratio not plotted")
df = df[np.isfinite(df["Odds Ratio"])]
df["Term"] = df["Term"].map(two_lines)

df = df.sort_values("Odds Ratio", ascending=False)

size_scale = 25
sizes = np.sqrt(df["Overlap_Count"]) * size_scale

def draw(ax, dpi):
    scatter = ax.scatter(
        df["Odds Ratio"],
        df["Term"],
        s=sizes if dpi == DPI else sizes / 4,
        c=df["neglog10_q"],
        cmap="Reds",
        vmin=COLOR_MIN,
        vmax=COLOR_MAX,
        alpha=SCATTER_ALPHA,
        edgecolor="black",
        linewidth=EDGE_WIDTH if dpi == DPI else 0.3
    )

    cbar = plt.colorbar(scatter, ax=ax, extend="max")
    if dpi == DPI:
        cbar.set_label("−log$_{10}$(adjusted P)", fontsize=FONT_SIZE_LABELS,
                       fontweight='bold', fontproperties=arial_font)
        cbar.set_ticks(range(COLOR_MIN, COLOR_MAX + 1))
        cbar.set_ticklabels([str(t) for t in range(COLOR_MIN, COLOR_MAX)] + [f"≥{COLOR_MAX}"])
        cbar.ax.tick_params(labelsize=FONT_SIZE_TICKS, width=SPINE_WIDTH)
        for label in cbar.ax.get_yticklabels():
            label.set_fontproperties(arial_font)
            label.set_fontsize(FONT_SIZE_TICKS)
        cbar.outline.set_linewidth(SPINE_WIDTH)

        legend_counts = [10, 50, 100]
        legend_sizes = np.sqrt(legend_counts) * size_scale
        handles = [
            ax.scatter([], [], s=s, edgecolor="black", facecolor="gray", alpha=SCATTER_ALPHA)
            for s in legend_sizes
        ]
        arial_font_legend = fm.FontProperties(fname=ARIAL, size=FONT_SIZE_LEGEND)
        legend = ax.legend(
            handles, [f"{c} genes" for c in legend_counts], title="Gene Count",
            loc="lower right", frameon=True, fancybox=False,
            labelspacing=0.5, borderpad=0.5, handletextpad=0.4,
            fontsize=14, prop=fm.FontProperties(fname=ARIAL, size=14),
            title_fontsize=14, edgecolor="black", framealpha=1
        )
        legend.get_title().set_fontproperties(arial_font_legend)
        legend.get_title().set_fontweight('bold')

    ax.invert_yaxis()
    ax.margins(y=0.15)

    if dpi == DPI:
        ax.set_xlabel("Odds ratio", fontsize=FONT_SIZE_LABELS, fontweight='bold', fontproperties=arial_font)
        ax.set_ylabel("")
        ax.text(-0.02, 1.02, "GO Biological Process", transform=ax.transAxes,
                fontsize=FONT_SIZE_LABELS, fontweight='bold', fontproperties=arial_font,
                ha='right', va='bottom')
        ax.tick_params(axis='both', which='major', labelsize=FONT_SIZE_TICKS,
                       width=SPINE_WIDTH, bottom=True, left=True)
        for label in ax.get_xticklabels():
            label.set_fontproperties(arial_font)
            label.set_fontsize(FONT_SIZE_TICKS)
        for label in ax.get_yticklabels():
            label.set_fontproperties(arial_font)
            label.set_fontsize(FONT_SIZE_TERMS)
    else:
        ax.set_xlabel("Odds ratio", fontsize=7)
        ax.tick_params(axis='both', which='major', labelsize=5)

    for spine in ['top', 'right']:
        ax.spines[spine].set_visible(False)
    for spine in ['left', 'bottom']:
        ax.spines[spine].set_linewidth(SPINE_WIDTH if dpi == DPI else 0.5)
        ax.spines[spine].set_color('black')

# --- Full-resolution save ---
fig, ax = plt.subplots(figsize=FIGURE_SIZE, dpi=DPI)
draw(ax, DPI)
plt.tight_layout()
plt.savefig('go_enrichment_constrained_2026-10.svg', dpi=DPI, bbox_inches='tight')
plt.savefig('go_enrichment_constrained_2026-10.png', dpi=DPI, bbox_inches='tight')
plt.close(fig)
print("saved go_enrichment_constrained_2026-10.svg/.png")

# --- Small inline preview ---
fig_prev, ax_prev = plt.subplots(figsize=(4, 3.5), dpi=50)
draw(ax_prev, 50)
plt.tight_layout()
plt.show()
plt.close(fig_prev)

In [ ]:
#@title Fig 4C
# Structure file for ChimeraX: AlphaFold DB v6 model of ZNF460 (Q14592) with B-factor -5 on every atom of its
# human-specific constrained residues (the coverage-filtered `hsc` table from the Fig 4A cell) and 0 elsewhere.
# Colour by B-factor in ChimeraX to reproduce the panel.
import urllib.request

UNIPROT = "Q14592"
pdb_name = f"AF-{UNIPROT}-F1-model_v6.pdb"
if not os.path.exists(pdb_name):
    urllib.request.urlretrieve(f"https://alphafold.ebi.ac.uk/files/{pdb_name}", pdb_name)

hsc_pos = set(hsc.loc[hsc["uniprot_id"] == UNIPROT, "uniprot_pos"].astype(int))
lines, marked = [], set()
for line in open(pdb_name):
    if line.startswith(("ATOM", "HETATM")):
        pos = int(line[22:26])
        b = -5.0 if line.startswith("ATOM") and pos in hsc_pos else 0.0
        if b:
            marked.add(pos)
        line = line[:60] + f"{b:6.2f}" + line[66:]
    lines.append(line)
out = f"ZNF460_AF-{UNIPROT}-F1_HSC_2026-10.pdb"
open(out, "w").writelines(lines)
print(f"{UNIPROT}: {len(hsc_pos)} human-specific constrained residues, {len(marked)} marked; saved {out}")

In [ ]:
#@title Fig 4D
# znf_finger_positions_2026-10.tsv: residues with ConSurf > 3 (eligible to be HSC) inside C2H2 zinc fingers
# (regex C-x2-4-C-x12-H-x3-5-H), with their offset from the first zinc histidine and HSC status:
# 276 proteins; recognition helix = positions 10-16 of the canonical finger (helix positions -1..6, defined a priori);
# rest of finger = all other finger positions. Bars = HSC / eligible residues; error bars = 95% CI from 1,000 bootstrap resamples
# of proteins. Fisher's exact test on the 2x2 table; rate-ratio CI from the same protein bootstrap.

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
from scipy.stats import fisher_exact
from pypalettes import load_cmap

DPI = 300
FONT_SIZE_LABELS = 18
FONT_SIZE_TICKS = 15
FONT_SIZE_TEXT = 14
SPINE_WIDTH = 1.2
N_BOOT = 1000
arial_font = fm.FontProperties(fname=ARIAL)

ef = pd.read_csv("znf_finger_positions_2026-10.tsv", sep="\t")
ef["helix"] = ef["offset"].between(-7, -1)

per = ef.groupby(["uniprot_id", "helix"])["hsc"].agg(["sum", "size"]).unstack(fill_value=0)
h_hit, h_n = per[("sum", True)].to_numpy(), per[("size", True)].to_numpy()
r_hit, r_n = per[("sum", False)].to_numpy(), per[("size", False)].to_numpy()
r_helix, r_rest = h_hit.sum() / h_n.sum(), r_hit.sum() / r_n.sum()

rng = np.random.default_rng(0)
bh, br, bratio = [], [], []
for _ in range(N_BOOT):
    i = rng.integers(0, len(per), len(per))
    a, b = h_hit[i].sum() / h_n[i].sum(), r_hit[i].sum() / r_n[i].sum()
    bh.append(a); br.append(b); bratio.append(a / b)
h_lo, h_hi = np.percentile(bh, [2.5, 97.5])
r_lo, r_hi = np.percentile(br, [2.5, 97.5])
ratio_lo, ratio_hi = np.percentile(bratio, [2.5, 97.5])

t = [[h_hit.sum(), h_n.sum() - h_hit.sum()], [r_hit.sum(), r_n.sum() - r_hit.sum()]]
odds, p = fisher_exact(t)
print(f"Recognition helix {h_hit.sum()}/{h_n.sum():,} = {100 * r_helix:.2f}% [{100 * h_lo:.2f}, {100 * h_hi:.2f}]")
print(f"Rest of finger    {r_hit.sum()}/{r_n.sum():,} = {100 * r_rest:.2f}% [{100 * r_lo:.2f}, {100 * r_hi:.2f}]")
print(f"OR {odds:.2f}, Fisher p {p:.1e}; rate ratio {r_helix / r_rest:.2f} [protein bootstrap 95% CI {ratio_lo:.2f}, {ratio_hi:.2f}]")

mant, expo = f"{p:.0e}".split("e")
p_str = f"{mant}×10$^{{{int(expo)}}}$"
cmap = load_cmap("AsteroidCity2")
colors = [cmap(0), "#BDBDBD"]

fig, ax = plt.subplots(figsize=(5, 5), dpi=DPI)
vals = [100 * r_helix, 100 * r_rest]
errs = [[100 * (r_helix - h_lo), 100 * (r_rest - r_lo)], [100 * (h_hi - r_helix), 100 * (r_hi - r_rest)]]
ax.bar([0, 1], vals, width=0.6, color=colors, edgecolor="black", linewidth=0.8, zorder=2)
ax.errorbar([0, 1], vals, yerr=errs, fmt="none", color="black", capsize=5, linewidth=1.2, zorder=3)

# significance bracket
ytop = 100 * max(h_hi, r_hi) * 1.08
ax.plot([0, 0, 1, 1], [ytop, ytop * 1.03, ytop * 1.03, ytop], color="black", linewidth=1.2)
ax.text(0.5, ytop * 1.05, f"OR = {odds:.2f}, p = {p_str}", ha="center", va="bottom",
        fontproperties=arial_font, fontsize=FONT_SIZE_TEXT)
ax.set_ylim(0, ytop * 1.22)

ax.set_xticks([0, 1])
ax.set_xticklabels([f"Recognition\nhelix", f"Rest of\nfinger"])
ax.set_xlim(-0.6, 1.6)
for lab in ax.get_xticklabels():
    lab.set_fontproperties(arial_font); lab.set_fontsize(FONT_SIZE_TICKS)
for lab in ax.get_yticklabels():
    lab.set_fontproperties(arial_font); lab.set_fontsize(FONT_SIZE_TICKS)
ax.set_ylabel("Human-specific constrained (%)", fontsize=FONT_SIZE_LABELS, fontproperties=arial_font)
ax.tick_params(axis="both", which="major", width=SPINE_WIDTH)
for s in ["top", "right"]:
    ax.spines[s].set_visible(False)
for s in ["left", "bottom"]:
    ax.spines[s].set_linewidth(SPINE_WIDTH)
ax.set_box_aspect(1)
plt.tight_layout()
plt.savefig("znf_helix_vs_rest_hsc_2026-10.png", dpi=DPI, bbox_inches="tight")
plt.savefig("znf_helix_vs_rest_hsc_2026-10.svg", dpi=DPI, bbox_inches="tight")
print("saved znf_helix_vs_rest_hsc_2026-10.png/.svg")
fig.set_dpi(60)   # small inline preview
plt.show()

In [ ]:
#@title Fig 5/6 inputs: ProteinGym scores of base and HuSC fine-tuned ESM2
# Per assay and model size: Spearman of the mean LLR over each variant's substitutions with the DMS score, for the
# base model and the HuSC fine-tuned LoRA adapter (finetuned_esm2_{size}_husc.pth). All assays are scored and labelled
# dev/test (proteingym_dev_test_split.csv); checkpoints were selected on dev, figures report test. If
# proteingym_scores_esm2_husc.csv exists, it is read instead of rescoring (rescoring needs a GPU).
!pip uninstall -y -q torchao

import os, re, glob, gc
import numpy as np
import pandas as pd
import torch
from scipy.stats import spearmanr
from transformers import EsmForMaskedLM, EsmTokenizer
from peft import LoraConfig, get_peft_model
from tqdm.auto import tqdm

SCORES_PATH = "proteingym_scores_esm2_husc.csv"
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

MODELS = {"8M":   "facebook/esm2_t6_8M_UR50D",
          "35M":  "facebook/esm2_t12_35M_UR50D",
          "150M": "facebook/esm2_t30_150M_UR50D",
          "650M": "facebook/esm2_t33_650M_UR50D"}
LORA = LoraConfig(task_type="FEATURE_EXTRACTION", r=8, lora_alpha=64, lora_dropout=0.1,
                  target_modules=["query", "key", "value", "output.dense"])
MAX_LEN = 1024

tokenizer = EsmTokenizer.from_pretrained(MODELS["8M"])
AA_LO, AA_HI = 4, 24
AA20 = [a for a, i in sorted(tokenizer.get_vocab().items(), key=lambda kv: kv[1]) if AA_LO <= i < AA_HI]
aa_to_idx = {a: i for i, a in enumerate(AA20)}
_MUT = re.compile(r"^([A-Z])(\d+)([A-Z])$")

PG_SUMMARY = pd.read_csv(f"DMS_substitutions.csv").drop_duplicates("DMS_id").set_index("DMS_id")
PG_SPLIT = pd.read_csv(f"proteingym_dev_test_split.csv").set_index("DMS_id")
DMS = {d: pd.read_csv(f"DMS/{d}.csv", usecols=["mutant", "DMS_score"])
       for d in tqdm(PG_SPLIT.index, desc="loading DMS assays")}

@torch.no_grad()
def llr_matrix(model, seq):
    """(L, 20) log-likelihood ratio of every substitution against the wild-type residue."""
    enc = tokenizer(seq, return_tensors="pt", truncation=True, max_length=MAX_LEN)
    logits = model(enc["input_ids"].to(device), attention_mask=enc["attention_mask"].to(device)).logits
    lp = torch.log_softmax(logits[0, 1:-1, AA_LO:AA_HI].float(), dim=-1)
    wt = torch.tensor([aa_to_idx.get(c, 0) for c in seq[:lp.size(0)]], device=device)
    return (lp - lp.gather(1, wt[:, None])).cpu().numpy()

def variant_scores(M, seq, mutants):
    """Mean LLR over each variant's substitutions (NaN if any substitution cannot be scored)."""
    out = np.full(len(mutants), np.nan)
    for i, mut in enumerate(mutants):
        vals = []
        for m in mut.split(":"):
            g = _MUT.match(m)
            if not g:
                vals = None; break
            wt_aa, p, mu = g.group(1), int(g.group(2)), g.group(3)
            if not (1 <= p <= M.shape[0]) or seq[p - 1] != wt_aa or mu not in aa_to_idx:
                vals = None; break
            vals.append(M[p - 1, aa_to_idx[mu]])
        if vals:
            out[i] = np.mean(vals)
    return out

def assay_spearman(M, dms_id):
    seq, dms = PG_SUMMARY.at[dms_id, "target_seq"], DMS[dms_id]
    s = variant_scores(M, seq, dms.mutant.astype(str).values)
    y = dms.DMS_score.values.astype(float)
    ok = ~np.isnan(s) & ~np.isnan(y)
    return spearmanr(s[ok], y[ok]).statistic if ok.sum() >= 10 else np.nan

def score_size(size):
    model = get_peft_model(EsmForMaskedLM.from_pretrained(MODELS[size]), LORA).to(device).eval()
    ckpt = f"finetuned_esm2_{size}_husc.pth"
    missing, unexpected = model.load_state_dict(torch.load(ckpt, map_location=device), strict=False)
    assert not unexpected and not [k for k in missing if "lora" in k], ckpt
    rows = {}
    for d in tqdm(PG_SPLIT.index, desc=size, leave=False):
        seq = PG_SUMMARY.at[d, "target_seq"]
        rows[d] = {f"ESM_{size}_ft_spearman": assay_spearman(llr_matrix(model, seq), d)}
        with model.disable_adapter():
            rows[d][f"ESM_{size}_base_spearman"] = assay_spearman(llr_matrix(model, seq), d)
    del model; gc.collect(); torch.cuda.empty_cache()
    return pd.DataFrame.from_dict(rows, orient="index")

if not os.path.exists(SCORES_PATH):
    parts = [score_size(size) for size in MODELS]
    scores = PG_SPLIT[["split", "taxon"]].join(PG_SUMMARY[["coarse_selection_type"]]).join(parts)
    scores.rename_axis("DMS_id").reset_index().to_csv(SCORES_PATH, index=False)

scores = pd.read_csv(SCORES_PATH)
for split, g in scores.groupby("split"):
    print(f"\n{split} ({len(g)} assays), mean Spearman")
    for size in MODELS:
        b, f = (g[f"ESM_{size}_{a}_spearman"].mean() for a in ["base", "ft"])
        print(f"  {size:>4s}  base {b:.3f}  HuSC fine-tuned {f:.3f} ({f - b:+.3f})")

In [ ]:
#@title Fig 5B
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
from scipy.stats import wilcoxon
import warnings

warnings.filterwarnings("ignore", category=DeprecationWarning)

# ============================================================================
# CONFIGURATION
# ============================================================================
FIGURE_SIZE = (7, 6)
DPI = 300
FONT_SIZE_LABELS = 18
FONT_SIZE_TICKS = 15
FONT_SIZE_LEGEND = 13
FONT_SIZE_TITLE = 13
FONT_SIZE_SIG = 16
SPINE_WIDTH = 1.2
BOX_WIDTH = 0.6
BOX_LINEWIDTH = 1.2
STRIP_SIZE = 4
STRIP_ALPHA = 0.5
STRIP_JITTER = 0.2

# Load Arial font
arial_font = fm.FontProperties(fname=ARIAL)

# ============================================================================
# DATA LOADING AND PROCESSING
# ============================================================================
# ProteinGym TEST assays only (checkpoints were selected on the dev split)
summary_df = pd.read_csv('proteingym_scores_esm2_husc.csv')
summary_df = summary_df[summary_df['split'] == 'test'].reset_index(drop=True)

# Map model sizes to their score columns
model_sizes = ['8M', '35M', '150M', '650M']
model_config = {size: (f'ESM_{size}_base_spearman', f'ESM_{size}_ft_spearman') for size in model_sizes}

# Convert wide → long format
melted_df = summary_df.melt(
    id_vars='taxon',
    value_vars=[c for size in model_sizes for c in model_config[size]],
    var_name='Model',
    value_name='Score'
)

# Extract model size and status (base or fine-tuned)
melted_df['Model_Size'] = melted_df['Model'].str.extract(r'(\d+M)')
melted_df['Status'] = melted_df['Model'].apply(
    lambda x: 'Fine-Tuned' if '_ft_' in x else 'Base'
)

# ============================================================================
# STATISTICAL ANALYSIS
# ============================================================================
significance_dict = {}

print("=" * 60)
print(f"STATISTICAL ANALYSIS - Wilcoxon Signed-Rank Test (ProteinGym test, n={len(summary_df)})")
print("=" * 60)

for size, (base_col, ft_col) in model_config.items():
    # Extract paired scores directly from summary_df
    base_scores = summary_df[base_col].dropna()
    ft_scores = summary_df[ft_col].dropna()

    # Ensure same proteins are used (in case of any missing values)
    valid_idx = base_scores.index.intersection(ft_scores.index)
    base_scores = base_scores.loc[valid_idx]
    ft_scores = ft_scores.loc[valid_idx]

    # Wilcoxon signed-rank test
    stat, p = wilcoxon(base_scores, ft_scores)

    # Print statistics
    print(f"\n{size} Model:")
    print(f"  Base median:       {base_scores.median():.4f}")
    print(f"  Fine-tuned median: {ft_scores.median():.4f}")
    print(f"  Wilcoxon p-value:  {p:.4e}")

    # Assign significance
    if p < 0.001:
        significance = '***'
    elif p < 0.01:
        significance = '**'
    elif p < 0.05:
        significance = '*'
    else:
        significance = 'ns'

    print(f"  Significance:      {significance}")
    significance_dict[size] = significance

print("=" * 60)

# ============================================================================
# COLOR PALETTE
# ============================================================================
# Nature-quality color scheme: Dark gray for base, muted red for fine-tuned
box_palette = ['#708090', '#d62728']    # base (dark gray), fine-tuned (muted red)
dot_palette = ['black', 'black']

# ============================================================================
# PLOTTING
# ============================================================================
fig, ax = plt.subplots(figsize=FIGURE_SIZE, dpi=DPI)

# Boxplot
sns.boxplot(
    data=melted_df,
    x='Model_Size',
    y='Score',
    hue='Status',
    palette=box_palette,
    fliersize=0,
    width=BOX_WIDTH,
    linewidth=BOX_LINEWIDTH,
    ax=ax
)

# Stripplot
sns.stripplot(
    data=melted_df,
    x='Model_Size',
    y='Score',
    hue='Status',
    dodge=True,
    size=STRIP_SIZE,
    alpha=STRIP_ALPHA,
    jitter=STRIP_JITTER,
    palette=dot_palette,
    edgecolor='black',
    linewidth=0.5,
    ax=ax
)

# ============================================================================
# LEGEND
# ============================================================================
# Remove duplicate legend entries
handles, labels = ax.get_legend_handles_labels()
legend = ax.legend(
    handles[:2],
    labels[:2],
    title="Model",
    fontsize=FONT_SIZE_LEGEND,
    loc='lower right',
    frameon=True,
    fancybox=False,
    edgecolor='black',
    framealpha=1
)

# Set Arial font for legend
for text in legend.get_texts():
    text.set_fontproperties(arial_font)
    text.set_fontsize(FONT_SIZE_LEGEND)
legend.get_title().set_fontproperties(arial_font)
legend.get_title().set_fontsize(FONT_SIZE_TITLE)

# ============================================================================
# AXES AND LABELS
# ============================================================================
# Axis labels
ax.set_xlabel("Model Size", fontsize=FONT_SIZE_LABELS, fontproperties=arial_font)
ax.set_ylabel("Spearman Correlation \n (Predictions vs. DMS fitness)", fontsize=FONT_SIZE_LABELS, fontproperties=arial_font)

# Tick labels
for label in ax.get_xticklabels() + ax.get_yticklabels():
    label.set_fontproperties(arial_font)
    label.set_fontsize(FONT_SIZE_TICKS)

# ============================================================================
# SPINES AND TICKS
# ============================================================================
# Remove top and right spines
for spine in ['top', 'right']:
    ax.spines[spine].set_visible(False)

# Format left and bottom spines
for spine in ['left', 'bottom']:
    ax.spines[spine].set_linewidth(SPINE_WIDTH)
    ax.spines[spine].set_color('black')

# Tick parameters
ax.tick_params(
    which='both',
    bottom=True,
    left=True,
    top=False,
    right=False,
    width=SPINE_WIDTH,
    color='black',
    labelsize=FONT_SIZE_TICKS
)

# ============================================================================
# SIGNIFICANCE ANNOTATIONS
# ============================================================================
y_max = melted_df['Score'].max()
y_min = melted_df['Score'].min()
y_offset = (y_max - y_min) * 0.05  # vertical spacing

for i, size in enumerate(model_sizes):
    sig = significance_dict[size]
    if sig != 'ns':
        ax.text(
            i,
            y_max + y_offset,
            sig,
            ha='center',
            va='bottom',
            fontsize=FONT_SIZE_SIG,
            fontproperties=arial_font,
            fontweight='bold'
        )

# ============================================================================
# FINALIZE AND SAVE
# ============================================================================
plt.tight_layout()

plt.savefig("fig5b_ESM_boxplot.svg", format="svg", dpi=DPI, bbox_inches='tight')

print("\n" + "=" * 60)
print("FIGURE SAVED:")
print("  - fig5b_ESM_boxplot.svg")
print("=" * 60)

plt.show()

In [ ]:
#@title Fig 5C

import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
from scipy.stats import wilcoxon
import warnings

warnings.filterwarnings("ignore", category=DeprecationWarning)

# ============================================================================
# CONFIGURATION
# ============================================================================
FIGURE_SIZE = (7, 6)
DPI = 300
FONT_SIZE_LABELS = 18
FONT_SIZE_TICKS = 15
FONT_SIZE_LEGEND = 13
FONT_SIZE_TITLE = 13
FONT_SIZE_SIG = 16
FONT_SIZE_PVAL = 12
SPINE_WIDTH = 1.2
BOX_WIDTH = 0.6
BOX_LINEWIDTH = 1.2
STRIP_SIZE = 4
STRIP_ALPHA = 0.5
STRIP_JITTER = 0.2

# Load Arial font
arial_font = fm.FontProperties(fname=ARIAL)

# ============================================================================
# DATA LOADING AND PROCESSING
# ============================================================================
# ProteinGym TEST assays only (checkpoints were selected on the dev split)
summary_df = pd.read_csv('proteingym_scores_esm2_husc.csv')
summary_df = summary_df[summary_df['split'] == 'test'].reset_index(drop=True)

# Define the desired order of taxa
taxon_order = ['Prokaryote', 'Eukaryote', 'Human', 'Virus']

# Melt data
melted_df = summary_df.melt(
    id_vars='taxon',
    value_vars=['ESM_650M_base_spearman', 'ESM_650M_ft_spearman'],
    var_name='Model',
    value_name='Score'
)

# Add Base / Fine-tuned labels
melted_df['Status'] = melted_df['Model'].apply(
    lambda x: 'Fine-Tuned' if '_ft_' in x else 'Base'
)

# Convert taxon to categorical with specified order
melted_df['taxon'] = pd.Categorical(
    melted_df['taxon'],
    categories=taxon_order,
    ordered=True
)

# ============================================================================
# COLOR PALETTE
# ============================================================================
# Publication-quality color scheme: Dark gray for base, red for fine-tuned
box_palette = ['#708090', '#d62728']     # base (dark gray), fine-tuned (red)
dot_palette = ['black', 'black']

# ============================================================================
# STATISTICAL ANALYSIS
# ============================================================================
significance_dict = {}
pvalue_dict = {}

print("=" * 60)
print("STATISTICAL ANALYSIS - Wilcoxon Signed-Rank Test")
print("=" * 60)

for taxon in taxon_order:
    # Filter data for this taxon
    subset = summary_df[summary_df['taxon'] == taxon]

    # Extract paired scores
    base_scores = subset['ESM_650M_base_spearman'].dropna()
    ft_scores = subset['ESM_650M_ft_spearman'].dropna()

    # Ensure same proteins are used
    valid_idx = base_scores.index.intersection(ft_scores.index)
    base_scores = base_scores.loc[valid_idx]
    ft_scores = ft_scores.loc[valid_idx]

    # Print statistics
    print(f"\n{taxon}:")
    print(f"  Base median:       {base_scores.median():.4f}")
    print(f"  Fine-tuned median: {ft_scores.median():.4f}")
    print(f"  n (paired obs):    {len(base_scores)}")

    # Check if we have enough data
    if len(base_scores) == 0 or len(ft_scores) == 0:
        print(f"  WARNING: Not enough data")
        significance_dict[taxon] = 'ns'
        pvalue_dict[taxon] = None
    elif len(base_scores) < 3:
        print(f"  WARNING: Very few observations")
        significance_dict[taxon] = 'ns'
        pvalue_dict[taxon] = None
    else:
        try:
            # Wilcoxon signed-rank test
            stat, p = wilcoxon(base_scores, ft_scores)
            pvalue_dict[taxon] = p

            # Assign significance
            if p < 0.001:
                significance = '***'
            elif p < 0.01:
                significance = '**'
            elif p < 0.05:
                significance = '*'
            else:
                significance = 'ns'

            significance_dict[taxon] = significance
            print(f"  Wilcoxon p-value:  {p:.4e}")
            print(f"  Significance:      {significance}")
        except Exception as e:
            print(f"  WARNING: Could not compute test: {e}")
            significance_dict[taxon] = 'ns'
            pvalue_dict[taxon] = None

print("=" * 60)

# ============================================================================
# PLOTTING
# ============================================================================
fig, ax = plt.subplots(figsize=FIGURE_SIZE, dpi=DPI)

# Boxplot
sns.boxplot(
    data=melted_df,
    x='taxon',
    y='Score',
    hue='Status',
    palette=box_palette,
    fliersize=0,
    width=BOX_WIDTH,
    linewidth=BOX_LINEWIDTH,
    order=taxon_order,
    ax=ax
)

# Stripplot
sns.stripplot(
    data=melted_df,
    x='taxon',
    y='Score',
    hue='Status',
    dodge=True,
    size=STRIP_SIZE,
    alpha=STRIP_ALPHA,
    jitter=STRIP_JITTER,
    palette=dot_palette,
    edgecolor='black',
    linewidth=0.5,
    order=taxon_order,
    ax=ax
)

# ============================================================================
# LEGEND
# ============================================================================
# Remove duplicate legend entries
handles, labels = ax.get_legend_handles_labels()
legend = ax.legend(
    handles[:2],
    labels[:2],
    title="Model",
    fontsize=FONT_SIZE_LEGEND,
    loc='lower left',
    frameon=True,
    fancybox=False,
    edgecolor='black',
    framealpha=1
)

# Set Arial font for legend
for text in legend.get_texts():
    text.set_fontproperties(arial_font)
    text.set_fontsize(FONT_SIZE_LEGEND)
legend.get_title().set_fontproperties(arial_font)
legend.get_title().set_fontsize(FONT_SIZE_TITLE)

# ============================================================================
# AXES AND LABELS
# ============================================================================
# Axis labels
ax.set_xlabel("Taxon", fontsize=FONT_SIZE_LABELS, fontproperties=arial_font)
ax.set_ylabel("Spearman Correlation \n (Predictions vs. DMS fitness)", fontsize=FONT_SIZE_LABELS, fontproperties=arial_font)

# Tick labels
for label in ax.get_xticklabels() + ax.get_yticklabels():
    label.set_fontproperties(arial_font)
    label.set_fontsize(FONT_SIZE_TICKS)

# ============================================================================
# SPINES AND TICKS
# ============================================================================
# Remove top and right spines
for spine in ['top', 'right']:
    ax.spines[spine].set_visible(False)

# Format left and bottom spines
for spine in ['left', 'bottom']:
    ax.spines[spine].set_linewidth(SPINE_WIDTH)
    ax.spines[spine].set_color('black')

# Tick parameters
ax.tick_params(
    which='both',
    bottom=True,
    left=True,
    top=False,
    right=False,
    width=SPINE_WIDTH,
    color='black',
    labelsize=FONT_SIZE_TICKS
)

# ============================================================================
# SIGNIFICANCE ANNOTATIONS
# ============================================================================
y_max = melted_df['Score'].max()
y_min = melted_df['Score'].min()
y_offset = (y_max - y_min) * 0.05

for i, taxon in enumerate(taxon_order):
    sig = significance_dict.get(taxon, 'ns')
    p_val = pvalue_dict.get(taxon)

    if sig != 'ns':
        # Show asterisks for significant results
        ax.text(
            i,
            y_max + y_offset,
            sig,
            ha='center',
            va='bottom',
            fontsize=FONT_SIZE_SIG,
            fontproperties=arial_font,
            fontweight='bold'
        )
    elif p_val is not None:
        # Show p-value for non-significant results
        ax.text(
            i,
            y_max + y_offset * 1.3,
            f'$\mathit{{p}}$={p_val:.3f}',
            ha='center',
            va='bottom',
            fontsize=FONT_SIZE_PVAL,
            fontproperties=arial_font
        )

# ============================================================================
# FINALIZE AND SAVE
# ============================================================================
plt.tight_layout()

# Save in multiple formats
plt.savefig("ESM_650M_boxplot_taxon.svg", format="svg", dpi=DPI, bbox_inches='tight')
plt.savefig("ESM_650M_boxplot_taxon.png", format="png", dpi=DPI, bbox_inches='tight')
plt.savefig("ESM_650M_boxplot_taxon.pdf", format="pdf", dpi=DPI, bbox_inches='tight')

print("\n" + "=" * 60)
print("FIGURE SAVED:")
print("  - ESM_650M_boxplot_taxon.svg")
print("  - ESM_650M_boxplot_taxon.png")
print("  - ESM_650M_boxplot_taxon.pdf")
print("=" * 60)

plt.show()

In [ ]:
#@title Fig 5D

import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
from scipy.stats import wilcoxon
import warnings

warnings.filterwarnings("ignore", category=DeprecationWarning)

# ============================================================================
# CONFIGURATION
# ============================================================================
FIGURE_SIZE = (7, 6)
DPI = 300
FONT_SIZE_LABELS = 18
FONT_SIZE_TICKS = 15
FONT_SIZE_LEGEND = 13
FONT_SIZE_TITLE = 13
FONT_SIZE_SIG = 16
FONT_SIZE_PVAL = 12
SPINE_WIDTH = 1.2
BOX_WIDTH = 0.6
BOX_LINEWIDTH = 1.2
STRIP_SIZE = 4
STRIP_ALPHA = 0.5
STRIP_JITTER = 0.2


# Load Arial font
arial_font = fm.FontProperties(fname=ARIAL)

# ============================================================================
# DATA LOADING AND PROCESSING
# ============================================================================
# ProteinGym TEST assays only (checkpoints were selected on the dev split)
summary_df = pd.read_csv('proteingym_scores_esm2_husc.csv')
summary_df = summary_df[summary_df['split'] == 'test'].reset_index(drop=True)

# Define category order
category_order = ['Stability', 'Activity', 'OrganismalFitness', 'Expression', 'Binding']

# Melt data
melted_df = summary_df.melt(
    id_vars='coarse_selection_type',
    value_vars=['ESM_650M_base_spearman', 'ESM_650M_ft_spearman'],
    var_name='Model',
    value_name='Score'
)

# Add Base / Fine-tuned labels
melted_df['Status'] = melted_df['Model'].apply(
    lambda x: 'Fine-Tuned' if '_ft_' in x else 'Base'
)

# Convert coarse_selection_type to categorical with specified order
melted_df['coarse_selection_type'] = pd.Categorical(
    melted_df['coarse_selection_type'],
    categories=category_order,
    ordered=True
)

# ============================================================================
# COLOR PALETTE
# ============================================================================
# Nature-quality color scheme: Dark gray for base, red for fine-tuned
box_palette = ['#708090', '#d62728']     # base (dark gray), fine-tuned (red)
dot_palette = ['black', 'black']

# ============================================================================
# STATISTICAL ANALYSIS
# ============================================================================
selection_types = category_order
significance_dict = {}
pvalue_dict = {}

print("=" * 60)
print("STATISTICAL ANALYSIS - Wilcoxon Signed-Rank Test")
print("=" * 60)

for sel_type in selection_types:
    # Filter data for this selection type
    subset = summary_df[summary_df['coarse_selection_type'] == sel_type]

    # Extract paired scores
    base_scores = subset['ESM_650M_base_spearman'].dropna()
    ft_scores = subset['ESM_650M_ft_spearman'].dropna()

    # Ensure same proteins are used
    valid_idx = base_scores.index.intersection(ft_scores.index)
    base_scores = base_scores.loc[valid_idx]
    ft_scores = ft_scores.loc[valid_idx]

    # Print statistics
    print(f"\n{sel_type}:")
    print(f"  Base median:       {base_scores.median():.4f}")
    print(f"  Fine-tuned median: {ft_scores.median():.4f}")
    print(f"  n (paired obs):    {len(base_scores)}")

    # Check if we have enough data
    if len(base_scores) == 0 or len(ft_scores) == 0:
        print(f"  WARNING: Not enough data")
        significance_dict[sel_type] = 'ns'
        pvalue_dict[sel_type] = None
    elif len(base_scores) < 3:
        print(f"  WARNING: Very few observations")
        significance_dict[sel_type] = 'ns'
        pvalue_dict[sel_type] = None
    else:
        try:
            # Wilcoxon signed-rank test
            stat, p = wilcoxon(base_scores, ft_scores)
            pvalue_dict[sel_type] = p

            # Assign significance
            if p < 0.001:
                significance = '***'
            elif p < 0.01:
                significance = '**'
            elif p < 0.05:
                significance = '*'
            else:
                significance = 'ns'

            significance_dict[sel_type] = significance
            print(f"  Wilcoxon p-value:  {p:.4e}")
            print(f"  Significance:      {significance}")
        except Exception as e:
            print(f"  WARNING: Could not compute test: {e}")
            significance_dict[sel_type] = 'ns'
            pvalue_dict[sel_type] = None

print("=" * 60)

# ============================================================================
# PLOTTING
# ============================================================================
fig, ax = plt.subplots(figsize=FIGURE_SIZE, dpi=DPI)

# Boxplot
sns.boxplot(
    data=melted_df,
    x='coarse_selection_type',
    y='Score',
    hue='Status',
    palette=box_palette,
    fliersize=0,
    width=BOX_WIDTH,
    linewidth=BOX_LINEWIDTH,
    order=category_order,
    ax=ax
)

# Stripplot
sns.stripplot(
    data=melted_df,
    x='coarse_selection_type',
    y='Score',
    hue='Status',
    dodge=True,
    size=STRIP_SIZE,
    alpha=STRIP_ALPHA,
    jitter=STRIP_JITTER,
    palette=dot_palette,
    edgecolor='black',
    linewidth=0.5,
    order=category_order,
    ax=ax
)

# ============================================================================
# LEGEND
# ============================================================================
# Remove duplicate legend entries
handles, labels = ax.get_legend_handles_labels()
legend = ax.legend(
    handles[:2],
    labels[:2],
    title="Model",
    fontsize=FONT_SIZE_LEGEND,
    loc='lower left',
    frameon=True,
    fancybox=False,
    edgecolor='black',
    framealpha=1
)

# Set Arial font for legend
for text in legend.get_texts():
    text.set_fontproperties(arial_font)
    text.set_fontsize(FONT_SIZE_LEGEND)
legend.get_title().set_fontproperties(arial_font)
legend.get_title().set_fontsize(FONT_SIZE_TITLE)

# ============================================================================
# AXES AND LABELS
# ============================================================================
# Axis labels
ax.set_xlabel("DMS Selection Type", fontsize=FONT_SIZE_LABELS, fontproperties=arial_font)
ax.set_ylabel("Spearman Correlation \n (Predictions vs. DMS fitness)", fontsize=FONT_SIZE_LABELS, fontproperties=arial_font)

# Format x-axis labels to split OrganismalFitness into two lines
xticklabels = []
for label in ax.get_xticklabels():
    text = label.get_text()
    if text == 'OrganismalFitness':
        xticklabels.append('Organismal\nFitness')
    else:
        xticklabels.append(text)

ax.set_xticklabels(xticklabels)

# Tick labels - apply font properties
for label in ax.get_xticklabels() + ax.get_yticklabels():
    label.set_fontproperties(arial_font)
    label.set_fontsize(FONT_SIZE_TICKS)

# ============================================================================
# SPINES AND TICKS
# ============================================================================
# Remove top and right spines
for spine in ['top', 'right']:
    ax.spines[spine].set_visible(False)

# Format left and bottom spines
for spine in ['left', 'bottom']:
    ax.spines[spine].set_linewidth(SPINE_WIDTH)
    ax.spines[spine].set_color('black')

# Tick parameters
ax.tick_params(
    which='both',
    bottom=True,
    left=True,
    top=False,
    right=False,
    width=SPINE_WIDTH,
    color='black',
    labelsize=FONT_SIZE_TICKS
)

# ============================================================================
# SIGNIFICANCE ANNOTATIONS
# ============================================================================
y_max = melted_df['Score'].max()
y_min = melted_df['Score'].min()
y_offset = (y_max - y_min) * 0.05

for i, sel_type in enumerate(selection_types):
    sig = significance_dict.get(sel_type, 'ns')
    p_val = pvalue_dict.get(sel_type)

    if sig != 'ns':
        # Show asterisks for significant results
        ax.text(
            i,
            y_max + y_offset,
            sig,
            ha='center',
            va='bottom',
            fontsize=FONT_SIZE_SIG,
            fontproperties=arial_font,
            fontweight='bold'
        )
    elif p_val is not None:
        # Show p-value for non-significant results
        ax.text(
            i,
            y_max + y_offset * 1.3,
            f'$\mathit{{p}}$={p_val:.3f}',
            ha='center',
            va='bottom',
            fontsize=FONT_SIZE_PVAL,
            fontproperties=arial_font
        )

# ============================================================================
# FINALIZE AND SAVE
# ============================================================================
plt.tight_layout()

# Save in multiple formats
plt.savefig("ESM_650M_boxplot_dms_selection_type.svg", format="svg", dpi=DPI, bbox_inches='tight')
plt.savefig("ESM_650M_boxplot_dms_selection_type.png", format="png", dpi=DPI, bbox_inches='tight')
plt.savefig("ESM_650M_boxplot_dms_selection_type.pdf", format="pdf", dpi=DPI, bbox_inches='tight')

print("\n" + "=" * 60)
print("FIGURE SAVED:")
print("  - ESM_650M_boxplot_dms_selection_type.svg")
print("  - ESM_650M_boxplot_dms_selection_type.png")
print("  - ESM_650M_boxplot_dms_selection_type.pdf")
print("=" * 60)

plt.show()

In [ ]:
#@title Fig 6 inputs: wild-type NLL and within/between-site decomposition for all model sizes
# One unmasked forward pass per ProteinGym target and model (base, HuSC fine-tuned) gives:
#  - wt_nll_esm2_{size}_husc.csv: per-residue NLL of the wild-type residue (full-vocabulary softmax)
#  - decomposition_esm2_{size}_husc.csv: per-assay Spearman of the variant score (mean LLR over substitutions),
#      overall, within-site (mean over sites with >= 3 single substitutions of Spearman(score, DMS)) and
#      between-site (Spearman of site-mean score vs site-mean DMS over the same sites), for both models.
# Requires the "Fig 5/6 inputs" cell above.

@torch.no_grad()
def forward(model, seq):
    """Per-residue wild-type NLL and the (L, 20) LLR matrix from one unmasked forward pass."""
    enc = tokenizer(seq, return_tensors="pt", truncation=True, max_length=MAX_LEN)
    ids = enc["input_ids"].to(device)
    lp = torch.log_softmax(model(ids, attention_mask=enc["attention_mask"].to(device)).logits[0, 1:-1].float(), dim=-1)
    nll = -lp.gather(1, ids[0, 1:-1, None])[:, 0]
    lp20 = torch.log_softmax(lp[:, AA_LO:AA_HI], dim=-1)
    wt = torch.tensor([aa_to_idx.get(c, 0) for c in seq[:lp20.size(0)]], device=device)
    return nll.cpu().numpy(), (lp20 - lp20.gather(1, wt[:, None])).cpu().numpy()

def parse_variants(seq, mutants, L):
    """List of (0-based position, amino-acid index) per variant; None if any substitution cannot be scored."""
    out = []
    for mut in mutants:
        parts = []
        for m in mut.split(":"):
            g = _MUT.match(m)
            if not g or not (1 <= int(g.group(2)) <= L) or seq[int(g.group(2)) - 1] != g.group(1) or g.group(3) not in aa_to_idx:
                parts = None; break
            parts.append((int(g.group(2)) - 1, aa_to_idx[g.group(3)]))
        out.append(parts)
    return out

def decompose(M, recs, y):
    s = np.array([np.mean([M[p, a] for p, a in r]) if r else np.nan for r in recs])
    ok = ~np.isnan(s) & ~np.isnan(y)
    out = dict(overall=spearmanr(s[ok], y[ok]).statistic if ok.sum() >= 10 else np.nan, within=np.nan, between=np.nan)
    single = ok & np.array([r is not None and len(r) == 1 for r in recs])
    d = pd.DataFrame(dict(pos=[r[0][0] for r, k in zip(recs, single) if k], s=s[single], y=y[single]))
    d = d.groupby("pos").filter(lambda t: len(t) >= 3)
    w = [spearmanr(t.s, t.y).statistic for _, t in d.groupby("pos") if t.s.nunique() > 1 and t.y.nunique() > 1]
    if w:
        out["within"] = float(np.mean(w))
    m = d.groupby("pos")[["s", "y"]].mean()
    if len(m) >= 5:
        out["between"] = spearmanr(m.s, m.y).statistic
    out["n_sites"] = len(m)
    return out

for size in MODELS:
    nll_path, dec_path = f"wt_nll_esm2_{size}_husc.csv", f"decomposition_esm2_{size}_husc.csv"
    if os.path.exists(nll_path) and os.path.exists(dec_path):
        continue
    model = get_peft_model(EsmForMaskedLM.from_pretrained(MODELS[size]), LORA).to(device).eval()
    ckpt = f"finetuned_esm2_{size}_husc.pth"
    missing, unexpected = model.load_state_dict(torch.load(ckpt, map_location=device), strict=False)
    assert not unexpected and not [k for k in missing if "lora" in k], ckpt
    out = {}
    for d in tqdm(PG_SPLIT.index, desc=size, leave=False):
        seq = PG_SUMMARY.at[d, "target_seq"]
        out[(d, "ft")] = forward(model, seq)
        with model.disable_adapter():
            out[(d, "base")] = forward(model, seq)
    del model; gc.collect(); torch.cuda.empty_cache()

    nll_rows, dec_rows = [], []
    for d in PG_SPLIT.index:
        seq, split = PG_SUMMARY.at[d, "target_seq"], PG_SPLIT.at[d, "split"]
        (nb, Mb), (nf, Mf) = out[(d, "base")], out[(d, "ft")]
        L = len(nb)
        nll_rows.append(pd.DataFrame(dict(DMS_id=d, split=split, position=np.arange(L), wt_aa=list(seq[:L]),
                                          nll_base=nb, nll_ft=nf)))
        recs = parse_variants(seq, DMS[d].mutant.astype(str).values, L)
        y = DMS[d].DMS_score.values.astype(float)
        row = dict(DMS_id=d, split=split)
        for arm, M in [("base", Mb), ("ft", Mf)]:
            for k, v in decompose(M, recs, y).items():
                row[k if k == "n_sites" else f"{k}_{arm}"] = v
        dec_rows.append(row)
    if not os.path.exists(nll_path):
        pd.concat(nll_rows).to_csv(nll_path, index=False)
    pd.DataFrame(dec_rows).to_csv(dec_path, index=False)
    print(f"{size}: written", flush=True)

for size in MODELS:
    dec = pd.read_csv(f"decomposition_esm2_{size}_husc.csv")
    dec = dec[dec.split == "test"]
    nll = pd.read_csv(f"wt_nll_esm2_{size}_husc.csv")
    nll = nll[nll.split == "test"].groupby("DMS_id")[["nll_base", "nll_ft"]].mean().mean()
    print(f"\n{size} (test, {len(dec)} assays)  mean WT NLL  base {nll.nll_base:.3f}  HuSC fine-tuned {nll.nll_ft:.3f}")
    for k in ["overall", "between", "within"]:
        print(f"  {k:8s} " + "  ".join(f"{a} {dec[f'{k}_{a}'].mean():.3f}" for a in ["base", "ft"]))

In [ ]:
#@title Fig 6A

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import matplotlib.font_manager as fm

# Configuration
FIGURE_SIZE = (8, 6)
DPI = 300
FONT_SIZE_LABELS = 18
FONT_SIZE_TICKS = 18
FONT_SIZE_LEGEND = 14
SPINE_WIDTH = 1.2

COLOR_BASE = '#708090'
COLOR_FT = '#d62728'

# Load Arial font
arial_font = fm.FontProperties(fname=ARIAL)

# Mean wild-type NLL per DMS dataset (unmasked forward pass), ProteinGym TEST assays only
wt_nll_df = pd.read_csv('wt_nll_esm2_650M_husc.csv')
summary_df = wt_nll_df[wt_nll_df['split'] == 'test'].groupby('DMS_id')[['nll_base', 'nll_ft']].mean()

base_means = summary_df['nll_base']
ft_means   = summary_df['nll_ft']

# Create figure
fig, ax = plt.subplots(figsize=FIGURE_SIZE, dpi=DPI)

# KDE plots (clipped at 0: NLL is non-negative)
sns.kdeplot(base_means, color=COLOR_BASE, fill=True, alpha=0.3, clip=(0, None),
            linewidth=2.5, label='650M Base Model', ax=ax)
sns.kdeplot(ft_means, color=COLOR_FT, fill=True, alpha=0.3, clip=(0, None),
            linewidth=2.5, label='650M Finetuned Model', ax=ax)

# Plot vertical lines for the median of each distribution
ax.axvline(base_means.median(), color=COLOR_BASE, linestyle='--',
           linewidth=2, alpha=0.8, label=f'Base median ({base_means.median():.2f})')
ax.axvline(ft_means.median(), color=COLOR_FT, linestyle='--',
           linewidth=2, alpha=0.8, label=f'Finetuned median ({ft_means.median():.2f})')

# Labels and formatting
ax.set_xlabel("Average NLL per DMS dataset", fontsize=FONT_SIZE_LABELS,
              fontweight='bold', fontproperties=arial_font)
ax.set_ylabel("Density", fontsize=FONT_SIZE_LABELS,
              fontweight='bold', fontproperties=arial_font)

# Tick formatting
ax.tick_params(axis='both', which='major', labelsize=FONT_SIZE_TICKS, width=SPINE_WIDTH, bottom=True, left=True)
for label in ax.get_xticklabels() + ax.get_yticklabels():
    label.set_fontproperties(arial_font)
    label.set_fontsize(FONT_SIZE_TICKS)

# Create Arial font with the desired size
arial_font_legend = fm.FontProperties(fname=ARIAL, size=FONT_SIZE_LEGEND)

# Legend
legend = ax.legend(fontsize=FONT_SIZE_LEGEND, frameon=True, fancybox=False,
                   edgecolor='black', framealpha=1, loc='upper right',
                   prop=arial_font_legend)

# Spines
for spine in ['top', 'right']:
    ax.spines[spine].set_visible(False)
for spine in ['left', 'bottom']:
    ax.spines[spine].set_linewidth(SPINE_WIDTH)
    ax.spines[spine].set_color('black')

plt.tight_layout()

plt.savefig('nll_distribution_comparison_650M.svg', dpi=DPI, bbox_inches='tight')
plt.savefig('nll_distribution_comparison_650M.png', dpi=DPI, bbox_inches='tight')
plt.show()

In [ ]:
#@title Fig 6B
# Wild-type NLL (unmasked forward pass, wt_nll_esm2_650M_husc.csv) averaged over each TEST assay's variation-intolerant
# (bottom 20% by site tolerance) and -tolerant (top 20%) sites, base vs HuSC fine-tuned ESM2-650M. P-values: base vs
# fine-tuned within each site group (two-sided Wilcoxon signed-rank, paired by assay).
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
import seaborn as sns
from scipy.stats import wilcoxon

FIGURE_SIZE = (8, 6)
DPI = 300
FONT_SIZE_LABELS = 18
FONT_SIZE_TICKS = 18
FONT_SIZE_TEXT = 16
FONT_SIZE_LEGEND = 14
SPINE_WIDTH = 1.2
MODEL_COLORS = {"Base": '#708090', "Fine-Tuned": '#d62728'}     # as Fig 5B / 6A
arial_font = fm.FontProperties(fname=ARIAL)

# Site tolerance = mean DMS_score of the single substitutions at a site (sites with >= MIN_SUBS singles); within each
# assay the bottom / top TAIL fraction of sites by tolerance are the variation intolerant / tolerant sites.
MIN_SUBS, MIN_SINGLES, MIN_SITES, TAIL = 3, 20, 10, 0.2
GROUPS = ["Variation Intolerant Sites", "Variation Tolerant Sites"]


def site_tolerance(dms_df):
    singles = dms_df[~dms_df["mutant"].str.contains(":")]
    if len(singles) < MIN_SINGLES:
        return None
    pos = singles["mutant"].str.extract(r"[A-Z](\d+)[A-Z]")[0].astype(int) - 1   # 1-indexed -> 0-indexed
    site = singles.groupby(pos)["DMS_score"].agg(["mean", "size"])
    return site.loc[site["size"] >= MIN_SUBS, "mean"]


nll_df = pd.read_csv(f"wt_nll_esm2_650M_husc.csv")
nll_df = nll_df[nll_df["split"] == "test"]
rows = []
for dms_id, nll in nll_df.groupby("DMS_id"):
    tol = site_tolerance(pd.read_csv(f"DMS/{dms_id}.csv", usecols=["mutant", "DMS_score"]))
    if tol is None:
        continue
    sites = nll.set_index("position").join(tol.rename("tolerance"), how="inner")
    if len(sites) < MIN_SITES:
        continue
    q = sites["tolerance"].rank(pct=True)
    for label, mask in [(GROUPS[0], q <= TAIL), (GROUPS[1], q > 1 - TAIL)]:
        sel = sites[mask]
        rows.append(dict(DMS_id=dms_id, group=label, n_sites=len(sel), nll_base=sel.nll_base.mean(),
                         nll_ft=sel.nll_ft.mean()))
wide = pd.DataFrame(rows)
wide.to_csv("wt_nll_tolerance_groups_650M.csv", index=False)
long = pd.concat([wide.assign(model=m, nll=wide[c]) for m, c in [("Base", "nll_base"), ("Fine-Tuned", "nll_ft")]])

print(f"{wide.DMS_id.nunique()} test assays")
tests = {}
for g in GROUPS:
    w = wide[wide.group == g]
    tests[g] = wilcoxon(w.nll_ft, w.nll_base)
    print(f"{g}: median WT NLL base {w.nll_base.median():.3f} -> fine-tuned {w.nll_ft.median():.3f}; "
          f"increased in {(w.nll_ft > w.nll_base).sum()}/{len(w)}; Wilcoxon p = {tests[g].pvalue:.2e}")
d = wide.pivot(index="DMS_id", columns="group", values="nll_ft") - wide.pivot(index="DMS_id", columns="group", values="nll_base")
r = wilcoxon(d[GROUPS[1]], d[GROUPS[0]])
print(f"Increase larger at tolerant than intolerant sites in {(d[GROUPS[1]] > d[GROUPS[0]]).sum()}/{len(d)} assays; "
      f"median increase {d[GROUPS[0]].median():+.3f} (intolerant) vs {d[GROUPS[1]].median():+.3f} (tolerant); p = {r.pvalue:.2e}")

fig, ax = plt.subplots(figsize=FIGURE_SIZE, dpi=DPI)
sns.violinplot(x="group", y="nll", hue="model", data=long, order=GROUPS, hue_order=list(MODEL_COLORS),
               palette=MODEL_COLORS, cut=0, inner="box", linewidth=SPINE_WIDTH + 0.5, ax=ax)
ymax, ymin = long.nll.max(), long.nll.min()
yoffset = 0.05 * (ymax - ymin)
for i, g in enumerate(GROUPS):                     # base vs fine-tuned within each site group (paired by assay)
    p = tests[g].pvalue
    p_text = "p < 0.001" if p < 0.001 else f"p = {p:.3f}"
    ax.plot([i - 0.2, i - 0.2, i + 0.2, i + 0.2], [ymax + yoffset, ymax + 1.3 * yoffset, ymax + 1.3 * yoffset, ymax + yoffset],
            color="black", linewidth=1)
    ax.text(i, ymax + 1.5 * yoffset, p_text, ha="center", va="bottom", fontsize=FONT_SIZE_TEXT,
            fontweight="bold", fontproperties=arial_font)
ax.set_ylim(bottom=0, top=ymax + 4 * yoffset)
ax.set_xlabel("")
ax.set_ylabel("Mean wild-type NLL per assay", fontsize=FONT_SIZE_LABELS, fontweight="bold", fontproperties=arial_font,
              labelpad=10)
ax.legend(title=None, frameon=False, loc="upper center", bbox_to_anchor=(0.5, -0.1), ncol=2,
          prop=fm.FontProperties(fname=ARIAL, size=FONT_SIZE_LEGEND))
ax.tick_params(axis="both", which="major", width=SPINE_WIDTH, color="black", bottom=True, left=True)
for label in ax.get_xticklabels() + ax.get_yticklabels():
    label.set_fontproperties(arial_font); label.set_fontsize(FONT_SIZE_TICKS)
for spine in ["top", "right"]:
    ax.spines[spine].set_visible(False)
for spine in ["left", "bottom"]:
    ax.spines[spine].set_linewidth(SPINE_WIDTH); ax.spines[spine].set_color("black")
plt.tight_layout()
plt.savefig("delta_nll_violin_variation_tolerance_650M.svg", dpi=DPI, bbox_inches='tight')
plt.savefig("delta_nll_violin_variation_tolerance_650M.png", dpi=DPI, bbox_inches='tight')
plt.show()

In [ ]:
#@title Fig 6C
# Spearman of the variant score with DMS, overall and split into between-position (site-mean score vs site-mean DMS) and
# within-position (mean over sites of the Spearman among substitutions at the site) components, for sites with >= 3 single
# substitutions; base vs HuSC fine-tuned ESM2-650M (TEST assays).
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
import seaborn as sns
from scipy.stats import wilcoxon

FIGURE_SIZE = (8, 6)
DPI = 300
FONT_SIZE_LABELS = 18
FONT_SIZE_TICKS = 18
FONT_SIZE_TEXT = 16
FONT_SIZE_LEGEND = 14
SPINE_WIDTH = 1.2
MODEL_COLORS = {"Base": '#708090', "Fine-Tuned": '#d62728'}     # as Fig 5B / 6A / 6B
COMPONENTS = {"overall": "Overall", "between": "Between-position", "within": "Within-position"}
arial_font = fm.FontProperties(fname=ARIAL)

dec = pd.read_csv(f"decomposition_esm2_650M_husc.csv")
dec = dec[dec["split"] == "test"]
long = pd.concat([pd.DataFrame({"DMS_id": dec.DMS_id, "component": label, "model": m, "spearman": dec[f"{k}_{arm}"]})
                  for k, label in COMPONENTS.items() for m, arm in [("Base", "base"), ("Fine-Tuned", "ft")]]).dropna()

tests = {}
print("=== Spearman with DMS, base vs HuSC fine-tuned ESM2-650M (test assays) ===")
for k, label in COMPONENTS.items():
    w = dec[[f"{k}_base", f"{k}_ft"]].dropna()
    d = w[f"{k}_ft"] - w[f"{k}_base"]
    tests[label] = wilcoxon(w[f"{k}_ft"], w[f"{k}_base"])
    print(f"{label:17s} n = {len(w)}  median base {w[f'{k}_base'].median():.3f} -> fine-tuned {w[f'{k}_ft'].median():.3f}  "
          f"Δ mean {d.mean():+.4f} median {d.median():+.4f}  improved {(d > 0).sum()}/{len(d)}  p = {tests[label].pvalue:.2e}")

fig, ax = plt.subplots(figsize=FIGURE_SIZE, dpi=DPI)
order = list(COMPONENTS.values())
sns.violinplot(x="component", y="spearman", hue="model", data=long, order=order, hue_order=list(MODEL_COLORS),
               palette=MODEL_COLORS, cut=0, inner="box", linewidth=SPINE_WIDTH + 0.5, ax=ax)
ax.axhline(0, color="black", linestyle="--", linewidth=1, zorder=0)
ymax, ymin = long.spearman.max(), long.spearman.min()
yoffset = 0.05 * (ymax - ymin)
for i, label in enumerate(order):                  # base vs fine-tuned within each component (paired by assay)
    p = tests[label].pvalue
    p_text = "p < 0.001" if p < 0.001 else f"p = {p:.2f}"
    ax.plot([i - 0.2, i - 0.2, i + 0.2, i + 0.2], [ymax + yoffset, ymax + 1.3 * yoffset, ymax + 1.3 * yoffset, ymax + yoffset],
            color="black", linewidth=1)
    ax.text(i, ymax + 1.5 * yoffset, p_text, ha="center", va="bottom", fontsize=FONT_SIZE_TEXT,
            fontweight="bold", fontproperties=arial_font)
ax.set_ylim(top=ymax + 4 * yoffset)
ax.set_xlabel("")
ax.set_ylabel("Spearman correlation\n(predictions vs. DMS)", fontsize=FONT_SIZE_LABELS, fontweight="bold",
              fontproperties=arial_font, labelpad=10)
ax.legend(title=None, frameon=False, loc="upper center", bbox_to_anchor=(0.5, -0.1), ncol=2,
          prop=fm.FontProperties(fname=ARIAL, size=FONT_SIZE_LEGEND))
ax.tick_params(axis="both", which="major", width=SPINE_WIDTH, color="black", bottom=True, left=True)
for label in ax.get_xticklabels() + ax.get_yticklabels():
    label.set_fontproperties(arial_font); label.set_fontsize(FONT_SIZE_TICKS)
for spine in ["top", "right"]:
    ax.spines[spine].set_visible(False)
for spine in ["left", "bottom"]:
    ax.spines[spine].set_linewidth(SPINE_WIDTH); ax.spines[spine].set_color("black")
plt.tight_layout()
plt.savefig("delta_spearman_decomposition_650M.svg", dpi=DPI, bbox_inches='tight')
plt.savefig("delta_spearman_decomposition_650M.png", dpi=DPI, bbox_inches='tight')
plt.show()

In [ ]:
#@title Fig 6D
# Per ProteinGym TEST assay: change (HuSC fine-tuned minus base ESM2-650M) in the Spearman between a site's wild-type NLL
# and its tolerance (x) vs the change in between-position Spearman (y; decomposition_esm2_650M_husc.csv, Fig 6C).
# Axes are zoomed to the bulk of assays; the two assays far outside (BIM binding Mcl-1, TDP-43) are drawn as
# arrows at the lower-left edge and labelled with their values. rho uses all assays.
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
from matplotlib.ticker import MultipleLocator
from scipy.stats import spearmanr, wilcoxon

FIGURE_SIZE = (8, 6)
DPI = 300
FONT_SIZE_LABELS = 18
FONT_SIZE_TICKS = 18
FONT_SIZE_TEXT = 16
FONT_SIZE_NOTE = 12
SPINE_WIDTH = 1.2
COLOR_FT = '#d62728'
XLIM, YLIM = (-0.075, 0.145), (-0.1, 0.125)       # bulk of assays; points outside are drawn at the edge
arial_font = fm.FontProperties(fname=ARIAL)

MIN_SUBS, MIN_SINGLES, MIN_SITES = 3, 20, 10                       # same site-tolerance definition as Fig 6B


def site_tolerance(dms_df):
    singles = dms_df[~dms_df["mutant"].str.contains(":")]
    if len(singles) < MIN_SINGLES:
        return None
    pos = singles["mutant"].str.extract(r"[A-Z](\d+)[A-Z]")[0].astype(int) - 1   # 1-indexed -> 0-indexed
    site = singles.groupby(pos)["DMS_score"].agg(["mean", "size"])
    return site.loc[site["size"] >= MIN_SUBS, "mean"]


nll_df = pd.read_csv(f"wt_nll_esm2_650M_husc.csv")
nll_df = nll_df[nll_df["split"] == "test"]
dec = pd.read_csv(f"decomposition_esm2_650M_husc.csv").set_index("DMS_id")
records = []
for dms_id, nll in nll_df.groupby("DMS_id"):
    tol = site_tolerance(pd.read_csv(f"DMS/{dms_id}.csv", usecols=["mutant", "DMS_score"]))
    if tol is None:
        continue
    sites = nll.set_index("position").join(tol.rename("tolerance"), how="inner")
    if len(sites) < MIN_SITES:
        continue
    coupling = {arm: spearmanr(sites[f"nll_{arm}"], sites["tolerance"]).statistic for arm in ["base", "ft"]}
    records.append(dict(DMS_id=dms_id, delta_coupling=coupling["ft"] - coupling["base"],
                        delta_between=dec.at[dms_id, "between_ft"] - dec.at[dms_id, "between_base"]))
df = pd.DataFrame(records).dropna()
df.to_csv("nll_tolerance_coupling_vs_between_gain_650M.csv", index=False)

r = spearmanr(df.delta_coupling, df.delta_between)
print(f"HuSC fine-tuned (n = {len(df)} assays): Δ coupling median {df.delta_coupling.median():+.4f}, > 0 in "
      f"{(df.delta_coupling > 0).sum()}/{len(df)}, Wilcoxon p = {wilcoxon(df.delta_coupling).pvalue:.2e}; "
      f"Δ between median {df.delta_between.median():+.4f}, > 0 in {(df.delta_between > 0).sum()}/{len(df)}; "
      f"Spearman(Δ coupling, Δ between) = {r.statistic:.3f}, p = {r.pvalue:.2e}")

inside = df.delta_coupling.between(*XLIM) & df.delta_between.between(*YLIM)
out = df[~inside].sort_values("delta_coupling")
print("Off-scale assays:\n" + out.to_string(index=False))

fig, ax = plt.subplots(figsize=FIGURE_SIZE, dpi=DPI)
ax.axhline(0, color="black", linestyle="--", linewidth=1, zorder=0)
ax.axvline(0, color="black", linestyle="--", linewidth=1, zorder=0)
ax.scatter(df[inside].delta_coupling, df[inside].delta_between, s=40, color=COLOR_FT, alpha=0.7, edgecolor="black",
           linewidth=0.5, zorder=2)
# Off-scale assays: arrows pointing toward their true position, stacked at the lower-left corner, labelled (x, y)
dx, dy = XLIM[1] - XLIM[0], YLIM[1] - YLIM[0]
for k, (_, row) in enumerate(out.iterrows()):
    x0, y0 = XLIM[0] + 0.035 * dx, YLIM[0] + (0.04 + 0.075 * k) * dy
    ax.annotate("", xy=(x0 - 0.03 * dx, y0 - 0.03 * dy), xytext=(x0, y0),
                arrowprops=dict(arrowstyle="-|>", color=COLOR_FT, lw=1.6), annotation_clip=False, zorder=3)
    ax.text(x0 + 0.012 * dx, y0, f"({row.delta_coupling:.2f}, {row.delta_between:.2f})".replace("-", "−"),
            ha="left", va="center", fontsize=FONT_SIZE_NOTE, fontproperties=arial_font, color=COLOR_FT)
ax.set_xlim(*XLIM); ax.set_ylim(*YLIM)
ax.xaxis.set_major_locator(MultipleLocator(0.05)); ax.yaxis.set_major_locator(MultipleLocator(0.05))
p_text = "p < 0.001" if r.pvalue < 0.001 else f"p = {r.pvalue:.3f}"
ax.text(0.03, 0.95, f"ρ = {r.statistic:.2f}, {p_text}", transform=ax.transAxes, ha="left", va="top",
        fontsize=FONT_SIZE_TEXT, fontweight="bold", fontproperties=arial_font, color=COLOR_FT)
ax.set_xlabel("Δ Spearman (WT NLL vs. site tolerance)", fontsize=FONT_SIZE_LABELS, fontweight="bold", fontproperties=arial_font)
ax.set_ylabel("Δ Between-position Spearman", fontsize=FONT_SIZE_LABELS, fontweight="bold", fontproperties=arial_font)
ax.tick_params(axis="both", which="major", width=SPINE_WIDTH, color="black", bottom=True, left=True)
for label in ax.get_xticklabels() + ax.get_yticklabels():
    label.set_fontproperties(arial_font); label.set_fontsize(FONT_SIZE_TICKS)
for spine in ["top", "right"]:
    ax.spines[spine].set_visible(False)
for spine in ["left", "bottom"]:
    ax.spines[spine].set_linewidth(SPINE_WIDTH); ax.spines[spine].set_color("black")
plt.tight_layout()
plt.savefig("nll_tolerance_coupling_vs_between_gain_650M.svg", dpi=DPI, bbox_inches='tight')
plt.savefig("nll_tolerance_coupling_vs_between_gain_650M.png", dpi=DPI, bbox_inches='tight')
plt.show()